# 🌍 Climate Trend Analyzer — Jupyter EDA Walkthrough

**Project:** Climate Trend Analyzer  
**Author:** [Your Name]  
**Location:** Mumbai, India | 1994–2024  

---

This notebook walks through the complete analysis pipeline:
1. Dataset overview
2. Exploratory Data Analysis (EDA)
3. Trend Analysis
4. Anomaly Detection
5. Forecasting
6. Insights

In [ ]:
# ─── Imports ───────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Dark theme for all plots
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#21262D',
    'axes.labelcolor':  '#E6EDF3',
    'xtick.color':      '#E6EDF3',
    'ytick.color':      '#E6EDF3',
    'text.color':       '#E6EDF3',
    'grid.color':       '#21262D',
    'grid.linestyle':   '--',
    'grid.linewidth':   0.5,
})

print('✅ Libraries imported successfully')

In [ ]:
# ─── 1. Load Data ──────────────────────────
import sys, os
sys.path.insert(0, '..')   # so we can import from src/

df = pd.read_csv('../data/climate_data_clean.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# ─── 2. Dataset Summary ────────────────────
print('=== DESCRIPTIVE STATISTICS ===')
display(df[['temperature','rainfall','humidity','wind_speed']].describe().round(2))

print('\n=== ANOMALY COUNTS ===')
print(df['anomaly'].value_counts())

print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

In [ ]:
# ─── 3. Temperature Trend ──────────────────
yearly = df.groupby('year')['temperature'].mean()
slope, intercept, r, p, se = stats.linregress(yearly.index, yearly.values)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(yearly.index, yearly.values.min()-0.2, yearly.values, alpha=0.1, color='#E84040')
ax.plot(yearly.index, yearly.values, color='#E84040', linewidth=2, marker='o', markersize=5, label='Yearly Avg')
ax.plot(yearly.index, slope*yearly.index+intercept, color='#FF8C00', linewidth=2.5,
        linestyle='--', label=f'Trend: {slope*10:+.3f}°C/decade')
ax.set_title('🌡️ Annual Mean Temperature Trend (1994–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Temperature (°C)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('../outputs/figures/nb_01_yearly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'R² = {r**2:.4f} | p-value = {p:.2e} | Warming = {slope*10:.3f}°C/decade')

In [ ]:
# ─── 4. Seasonal Analysis ──────────────────
seasons = ['Winter','Spring','Monsoon','Autumn']
data    = [df[df['season']==s]['temperature'].dropna().values for s in seasons]
colors  = ['#74B9FF','#55EFC4','#FDCB6E','#E17055']

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(data, patch_artist=True, notch=True,
                medianprops=dict(color='white', linewidth=2))
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.8)
ax.set_xticklabels(seasons)
ax.set_title('🍂 Seasonal Temperature Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (°C)')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5. Rainfall Heatmap ───────────────────
pivot = df.groupby(['year','month'])['rainfall'].sum().unstack()

fig, ax = plt.subplots(figsize=(15, 8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax.set_xticks(range(12)); ax.set_xticklabels(months)
plt.colorbar(im, ax=ax, label='Total Rainfall (mm)')
ax.set_title('🌧️ Monthly Rainfall Heatmap (1994–2024)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 6. Anomaly Detection (Isolation Forest) ──
features = ['temperature','rainfall','humidity','wind_speed']
X = df[features].fillna(df[features].median())
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

clf  = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
pred = clf.fit_predict(X_sc)
df['if_anomaly'] = (pred == -1)

print(f'Isolation Forest anomalies: {df["if_anomaly"].sum():,} ({df["if_anomaly"].mean()*100:.2f}%)')
display(df[df['if_anomaly']].groupby('year').size().rename('Anomaly Days'))

In [ ]:
# ─── 7. Forecast (Linear Regression) ──────
from sklearn.linear_model import LinearRegression

yearly = df.groupby('year')['temperature'].mean().reset_index()
X_yr = yearly['year'].values.reshape(-1,1)
y_yr = yearly['temperature'].values

model = LinearRegression().fit(X_yr, y_yr)
future = np.arange(2025, 2035).reshape(-1,1)
fc     = model.predict(future)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(yearly['year'], y_yr, color='#3A7BD5', linewidth=2, marker='o', markersize=5, label='Historical')
ax.plot(future.flatten(), fc, color='#E84040', linewidth=2.5, marker='s', linestyle='--', label='Forecast (2025–2034)')
ax.axvline(2024, color='white', linewidth=1, linestyle=':', alpha=0.5)
ax.set_title('🔮 Temperature Forecast — Linear Regression', fontsize=13, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Temperature (°C)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

fc_df = pd.DataFrame({'Year': future.flatten(), 'Predicted_Temp': fc.round(3)})
print(fc_df.to_string(index=False))

In [ ]:
# ─── 8. Final Key Insights ─────────────────
yearly_avg = df.groupby('year')['temperature'].mean()
sl, ic, r_val, p_val, _ = stats.linregress(yearly_avg.index, yearly_avg.values)

print('='*55)
print('   CLIMATE TREND ANALYZER — KEY FINDINGS')
print('='*55)
print(f'  30-yr Mean Temperature  : {df["temperature"].mean():.2f}°C')
print(f'  Warming rate            : {sl*10:+.3f}°C/decade')
print(f'  R²                      : {r_val**2:.4f}')
print(f'  p-value                 : {p_val:.2e} {"✅ Significant" if p_val < 0.05 else "❌"}')
print(f'  Hottest year            : {yearly_avg.idxmax()} ({yearly_avg.max():.2f}°C)')
print(f'  Monsoon rain share      : {df[df["month"].between(6,9)]["rainfall"].sum()/df["rainfall"].sum()*100:.1f}%')
print(f'  Anomaly days (injected) : {(df["anomaly"]!="Normal").sum()}')
print(f'  Forecast 2030           : {sl*2030+ic:.2f}°C')
print(f'  Forecast 2050           : {sl*2050+ic:.2f}°C')
print('='*55)